# Cuaderno de Práctica: Funciones en Python para Data Science
Este cuaderno contiene ejercicios diseñados para aplicar los conceptos teóricos de la Sesión 4. 
Intenta resolver cada celda de código por tu cuenta antes de mirar las soluciones.

In [39]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 7
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.4f}".format)

print("Versión de pandas:", pd.__version__)

Versión de pandas: 2.3.3


## Ejercicio 1: Filtrado Complejo y Creación Condicional de Columnas 🕵️‍♂️
* **Contexto:** En un dataset de transacciones bancarias, necesitas identificar operaciones sospechosas de fraude sin usar bucles lentos.
* **Tarea:** Crea un DataFrame ficticio con columnas `monto`, `tipo_operacion` y `pais_origen`.
* **Requisito:** 1. Usa máscaras booleanas para aislar las transacciones donde el `monto` supere los $10,000 **Y** el `tipo_operacion` sea "Transferencia Internacional". 
  2. Crea una nueva columna llamada `alerta_fraude`. Utiliza una función `lambda` con `.apply()` (o vectorización directa) para asignar `True` a las filas que cumplen la condición y `False` al resto.

In [13]:
n_clientes = 10000
clientes = pd.DataFrame({
    "id_cliente": np.arange(0 ,0 + n_clientes),
    "monto": rng.uniform(0,100000, size=n_clientes),
    "tipo_operacion": rng.choice(["Transferencia Nacional","Transferencia Internacional"],size=n_clientes,p=[0.7,0.3]),
    "pais_origen": rng.choice(["Perú","Alemania","Argentina","EEUU","Bolivia","Brasil","Ecuador","Chile"],size=n_clientes)
})

clientes.head()

,id_cliente,monto,tipo_operacion,pais_origen
0,0,"18,803.4742",Transferencia Internacional,Chile
1,1,"2,236.1777",Transferencia Internacional,Brasil
2,2,"32,318.8125",Transferencia Nacional,Perú
3,3,"22,637.8016",Transferencia Nacional,Ecuador
4,4,"53,950.9200",Transferencia Internacional,Argentina


In [43]:
mask_monto = clientes.monto > 10000 
mask_transferencia= clientes.tipo_operacion == "Transferencia Internacional"
mask = mask_monto | mask_transferencia
mask_monto,mask_transferencia,mask

(0        True
 1       False
 2        True
 3        True
 4        True
         ...  
 9995     True
 9996     True
 9997     True
 9998     True
 9999     True
 Name: monto, Length: 10000, dtype: bool,
 0        True
 1        True
 2       False
 3       False
 4        True
         ...  
 9995    False
 9996     True
 9997    False
 9998    False
 9999    False
 Name: tipo_operacion, Length: 10000, dtype: bool,
 0       True
 1       True
 2       True
 3       True
 4       True
         ... 
 9995    True
 9996    True
 9997    True
 9998    True
 9999    True
 Length: 10000, dtype: bool)

In [49]:
clientes["Alerta_Fraude"] = mask
clientes.head()

,id_cliente,monto,tipo_operacion,pais_origen,Alerta_Fraude
0,0,"18,803.4742",Transferencia Internacional,Chile,True
1,1,"2,236.1777",Transferencia Internacional,Brasil,True
2,2,"32,318.8125",Transferencia Nacional,Perú,True
3,3,"22,637.8016",Transferencia Nacional,Ecuador,True
4,4,"53,950.9200",Transferencia Internacional,Argentina,True


## Ejercicio 2: Agrupación Multicanal y Agregaciones (`groupby` y `agg`) 📊
* **Contexto:** Analizas las ventas mensuales de una tienda por departamentos en distintas regiones y necesitas métricas combinadas.
* **Tarea:** Dado un DataFrame con `region`, `categoria_producto`, `ventas` y `costos`.
* **Requisito:** 1. Agrupa los datos utilizando una lista doble: por `region` y luego por `categoria_producto`. 
  2. Utiliza el método `.agg()` pasando un diccionario para calcular simultáneamente la suma total de `ventas` y el promedio de `costos`. 
  3. Resetea el índice (`reset_index`) para que el resultado vuelva a ser un DataFrame plano.

In [51]:
n_ventas = 10000
data =pd.DataFrame({
    "id":np.arange(0, 0 + n_ventas),
    "region":rng.choice(["Lima","Ayacucho","Tarapoto","Cuzo","Huáraz","Trujillo"],size=n_ventas),
    "categoria_producto":rng.choice(["Alimento","Juguete","Electródomestico","Limpieza"],size=n_ventas),
    "costos":rng.uniform(10,900,size=n_ventas)
})
data.head()

,id,region,categoria_producto,costos
0,0,Cuzo,Alimento,264.0705
1,1,Lima,Limpieza,487.1039
2,2,Trujillo,Juguete,148.6854
3,3,Trujillo,Limpieza,177.5622
4,4,Huáraz,Limpieza,41.3081


In [54]:
resumen = data.groupby(["region","categoria_producto"]).agg({
    'id': 'count',
    'costos': 'mean'
})
resumen

id   costos
region   categoria_producto              
Ayacucho Alimento            424 485.0214
         Electródomestico    403 447.8511
         Juguete             409 456.0995
         Limpieza            405 448.3656
Cuzo     Alimento            434 451.3336
         Electródomestico    414 464.7129
         Juguete             432 450.7387
         Limpieza            416 454.0592
Huáraz   Alimento            402 432.9262
         Electródomestico    433 443.6333
         Juguete             392 444.6416
         Limpieza            444 450.6413
Lima     Alimento            457 461.6923
         Electródomestico    446 442.1735
         Juguete             401 440.9796
         Limpieza            404 461.6741
Tarapoto Alimento            394 458.0774
         Electródomestico    446 433.0672
         Juguete             392 463.3753
         Limpieza            442 479.5337
Trujillo Alimento            418 480.9640
         Electródomestico    401 464.2639
         Juguete             377 469.9519
         Limpieza            414 451.6767

In [55]:
resumen.reset_index()

,region,categoria_producto,id,costos
0,Ayacucho,Alimento,424,485.0214
1,Ayacucho,Electródomestico,403,447.8511
2,Ayacucho,Juguete,409,456.0995
3,Ayacucho,Limpieza,405,448.3656
4,Cuzo,Alimento,434,451.3336
5,Cuzo,Electródomestico,414,464.7129
6,Cuzo,Juguete,432,450.7387
7,Cuzo,Limpieza,416,454.0592
8,Huáraz,Alimento,402,432.9262
9,Huáraz,Electródomestico,433,443.6333


## Ejercicio 3: Combinación Robusta de Fuentes (`merge` y `fillna`) 🔗
* **Contexto:** Tienes la base de datos dividida: por un lado la información de los usuarios y por otro su historial de compras mensual.
* **Tarea:** Crea `df_usuarios` (con `id_usuario`, `nombre`) y `df_compras` (con `id_compra`, `id_usuario`, `monto_total`). Ojo: incluye en `df_usuarios` a personas que no están en `df_compras`.
* **Requisito:** 1. Realiza un `Left Join` usando `pd.merge()` para mantener a todos los usuarios, incluso aquellos que no han comprado nada. 
  2. Identifica la columna de compras resultante y rellena los valores nulos (`NaN`) con `0.0` utilizando el método `.fillna()`.

In [64]:
n_usuarios = 10000
df_usuarios = pd.DataFrame({
    "id_usuario":np.arange(0,0+n_usuarios),
    "nombre":rng.choice(["Ana", "Carlos", "María", "Jorge", "Lucía", "Daniel", "Sofía", "Miguel", "Elena", "David", "Carmen", "Alejandro", "Isabel", "Pablo", "Laura"],size=n_usuarios)
})
df_usuarios.loc[rng.choice(clientes.index, size=50, replace=False), "nombre"] = np.nan

df_compras = pd.DataFrame({
    "id_compras":np.arange(0,0+n_usuarios),
    "id_usuario":rng.integers(0,0+n_usuarios,size=n_usuarios),
    "monto_total":rng.uniform(0,10000,size=n_usuarios)
})
df_compras.loc[rng.choice(clientes.index, size=50, replace=False), "id_usuario"] = np.nan


In [66]:
df = pd.merge(df_usuarios,df_compras,on ="id_usuario",how='left')
df.head()

,id_usuario,nombre,id_compras,monto_total
0,0,Elena,"4,590.0000","5,226.3939"
1,0,Elena,"6,159.0000","1,854.6557"
2,1,David,NaN,NaN
3,2,María,"6,236.0000","4,208.3919"
4,3,Pablo,"4,814.0000",993.1119


In [69]:
df.fillna(0)

,id_usuario,nombre,id_compras,monto_total
0,0,Elena,"4,590.0000","5,226.3939"
1,0,Elena,"6,159.0000","1,854.6557"
2,1,David,0.0000,0.0000
3,2,María,"6,236.0000","4,208.3919"
4,3,Pablo,"4,814.0000",993.1119
...,...,...,...,...
13690,9997,Lucía,"5,596.0000","1,543.8058"
13691,9998,Daniel,"6,146.0000","4,848.8559"
13692,9998,Daniel,"9,683.0000","5,427.0565"
13693,9999,Ana,"4,196.0000","5,081.3361"


## Ejercicio 4: Reestructuración y Tablas Dinámicas (`pivot_table`) 📈
* **Contexto:** Necesitas presentar a gerencia un resumen ejecutivo del rendimiento cruzado de tus vendedores por trimestre.
* **Tarea:** Trabajas con un DataFrame que tiene `vendedor`, `trimestre` (Q1, Q2, etc.), `sucursal` y `monto_vendido`.
* **Requisito:** Utiliza `pd.pivot_table()` para crear una matriz donde las **filas** sean los nombres de los vendedores, las **columnas** sean los trimestres, y los **valores** sean la suma del `monto_vendido`. Asegúrate de manejar los posibles valores nulos (vendedores que no operaron en un trimestre específico) seteando el parámetro `fill_value=0`.

In [43]:
N_operaciones = 1000
df = pd.DataFrame({
    "vendedor":rng.choice(["Ana", "Carlos", "Beatriz", "Daniel", "Elena", "Fernando", "Gabriela", "Héctor", "Isabel", "Javier"], size = N_operaciones),
    "Trimestre":rng.choice(['Q1','Q2','Q3'],size=N_operaciones),
    "sucursal":rng.choice(["Lima","Trujillo","Huáraz","Cajarmaca","Arequipa"]),
    "monto_vendido":rng.uniform(0,10000,size=N_operaciones)
})
df.head()

,vendedor,Trimestre,sucursal,monto_vendido
0,Javier,Q2,Trujillo,"6,262.3328"
1,Ana,Q1,Trujillo,"7,052.5174"
2,Gabriela,Q2,Trujillo,"6,947.7649"
3,Ana,Q2,Trujillo,"8,749.2076"
4,Elena,Q1,Trujillo,"6,817.9146"


In [47]:
reporte = pd.pivot_table(
    df,
    index="vendedor",
    columns="Trimestre",
    values="monto_vendido",
    aggfunc="sum",
    fill_value=0,
    margins=True
).round(4)
reporte

Trimestre,Q1,Q2,Q3,All
vendedor,,,,
Ana,"192,706.7602","136,891.9737","190,826.0369","520,424.7708"
Beatriz,"126,938.4613","198,426.7242","237,079.3971","562,444.5827"
Carlos,"183,895.9401","142,984.5831","209,734.8292","536,615.3524"
Daniel,"149,427.7042","100,899.5754","113,500.8974","363,828.1770"
Elena,"162,570.4323","86,530.9848","153,511.8806","402,613.2977"
Fernando,"207,284.0519","186,100.4342","136,043.1476","529,427.6336"
Gabriela,"164,543.8289","253,277.4411","216,951.9262","634,773.1962"
Héctor,"124,098.0007","142,536.2762","137,804.1593","404,438.4362"
Isabel,"132,134.8390","172,974.6517","177,078.9283","482,188.4191"


## Ejercicio 5: Limpieza Estricta de Strings (`.str`) 🧹
* **Contexto:** Los datos ingresados manualmente suelen tener errores tipográficos, mayúsculas revueltas y espacios extra que arruinan los cruces de datos.
* **Tarea:** Tienes una columna `correo_electronico` en tu DataFrame con valores sucios (ej. `"  JUAN@gmail.com "`, `"ana.perez @ EMPRESA.net "`).
* **Requisito:** Encadena métodos de acceso `.str` para: 
  1. Eliminar los espacios en blanco a los lados (`strip`).
  2. Eliminar espacios internos reemplazándolos por nada (`replace`).
  3. Convertir todo a minúsculas (`lower`). 
  4. Finalmente, crea una máscara para mantener solo las filas cuyos correos terminen explícitamente en `".com"`.

In [7]:
#Creamos el Data Frame:
import pandas as pd
import numpy as np

# Semilla para reproducibilidad
np.random.seed(42)

# Generar 100 nombres aleatorios simples
nombres = [f"usuario{i}" for i in range(1, 101)]
dominios = ["gmail.com", "yahoo.com", "empresa.net", "hotmail.com", "outlook.com"]

# Lista para almacenar los correos "sucios"
correos_sucios = []

for nombre in nombres:
    dominio = np.random.choice(dominios)
    # Generar variaciones de suciedad
    tipo_suciedad = np.random.choice([
        "espacios_ext",      # " nombre@dominio.com "
        "mayusculas",        # " NOMBRE@DOMINIO.COM "
        "espacio_at",        # " nombre @ dominio.com "
        "doble_sucia",       # "  NOMBRE @ DOMINIO.COM  "
        "limpio"             # Caso raro sin suciedad
    ])
    
    correo_base = f"{nombre}@{dominio}"
    
    if tipo_suciedad == "espacios_ext":
        correo = f" {correo_base} "
    elif tipo_suciedad == "mayusculas":
        correo = correo_base.upper()
    elif tipo_suciedad == "espacio_at":
        # Insertar espacio antes del @ y a veces después del dominio
        parts = correo_base.split("@")
        correo = f" {parts} @ {parts} "
    elif tipo_suciedad == "doble_sucia":
        correo = f"  {correo_base.upper()}  "
    else:
        correo = correo_base
    
    correos_sucios.append(correo)

# Crear el DataFrame
df = pd.DataFrame({
    "id": range(1, 101),
    "nombre": nombres,
    "correo_electronico": correos_sucios
})

# Mostrar las primeras filas para verificar la "suciedad"
print(df.head(10))
print(f"\nDimensiones del DataFrame: {df.shape}")

   id     nombre                                 correo_electronico
0   1   usuario1                               usuario1@hotmail.com
1   2   usuario2                               usuario2@empresa.net
2   3   usuario3                               USUARIO3@OUTLOOK.COM
3   4   usuario4   ['usuario4', 'empresa.net'] @ ['usuario4', 'e...
4   5   usuario5                               usuario5@empresa.net
5   6   usuario6   ['usuario6', 'hotmail.com'] @ ['usuario6', 'h...
6   7   usuario7                               USUARIO7@OUTLOOK.COM
7   8   usuario8                               USUARIO8@HOTMAIL.COM
8   9   usuario9                               usuario9@hotmail.com
9  10  usuario10                              USUARIO10@GMAIL.COM  

Dimensiones del DataFrame: (100, 3)


In [33]:
#Usamos .str.replace para reemplazar valores, '\s+' sirve para reemplazar los saltos en la misma línea.
#Usamos el [ Aquí ponemos los símbolo que queremos eliminar]
df["Correo_Electrónico_Limpio"] = df["correo_electronico"].str.strip().str.replace(r'[\[\]\"\'\,,\s+]','',regex=True).str.lower()
df.head(10)

,id,nombre,correo_electronico,Correo_Electrónico_Limpio
0,1,usuario1,usuario1@hotmail.com,usuario1@hotmail.com
1,2,usuario2,usuario2@empresa.net,usuario2@empresa.net
2,3,usuario3,USUARIO3@OUTLOOK.COM,usuario3@outlook.com
3,4,usuario4,"['usuario4', 'empresa.net'] @ ['usuario4', 'e...",usuario4empresa.net@usuario4empresa.net
4,5,usuario5,usuario5@empresa.net,usuario5@empresa.net
5,6,usuario6,"['usuario6', 'hotmail.com'] @ ['usuario6', 'h...",usuario6hotmail.com@usuario6hotmail.com
6,7,usuario7,USUARIO7@OUTLOOK.COM,usuario7@outlook.com
7,8,usuario8,USUARIO8@HOTMAIL.COM,usuario8@hotmail.com
8,9,usuario9,usuario9@hotmail.com,usuario9@hotmail.com
9,10,usuario10,USUARIO10@GMAIL.COM,usuario10@gmail.com


In [35]:
mask = df["Correo_Electrónico_Limpio"].str.endswith('.com') #Filtramos las filas que terminan con .com
df_limpio = df[mask].reset_index(drop = True) #Aplicar la máscara para quedarte solo con esas filas
df_limpio

,id,nombre,correo_electronico,Correo_Electrónico_Limpio
0,1,usuario1,usuario1@hotmail.com,usuario1@hotmail.com
1,3,usuario3,USUARIO3@OUTLOOK.COM,usuario3@outlook.com
2,6,usuario6,"['usuario6', 'hotmail.com'] @ ['usuario6', 'h...",usuario6hotmail.com@usuario6hotmail.com
3,7,usuario7,USUARIO7@OUTLOOK.COM,usuario7@outlook.com
4,8,usuario8,USUARIO8@HOTMAIL.COM,usuario8@hotmail.com
...,...,...,...,...
76,95,usuario95,"['usuario95', 'gmail.com'] @ ['usuario95', 'g...",usuario95gmail.com@usuario95gmail.com
77,96,usuario96,usuario96@yahoo.com,usuario96@yahoo.com
78,97,usuario97,USUARIO97@HOTMAIL.COM,usuario97@hotmail.com
79,98,usuario98,"['usuario98', 'hotmail.com'] @ ['usuario98', ...",usuario98hotmail.com@usuario98hotmail.com


In [36]:
#POR AHORA NOS FALTA LIMPIAR MÁS, ASEGURARNOS QUE EN LOS Q SE REPITEN SE MANTIENEN CON EL MISMO USARIO
#ADEMÁS DE PRIMERO SEPARALOS POR EN @ , LUEGO QUEDARNOS CON EL DE LA DERECHA Y AGREGA EL @

## Ejercicio 6: Ventanas Móviles para Tendencias (`rolling` y `shift`) 📉
* **Contexto:** En finanzas y series de tiempo, calcular medias móviles es vital para suavizar el ruido diario e identificar tendencias.
* **Tarea:** Crea un DataFrame con una columna `fecha` (puedes usar `pd.date_range`) y una columna `precio_cierre`.
* **Requisito:** 1. Crea una nueva columna llamada `media_movil_3d` utilizando el método `.rolling(window=3).mean()` sobre la columna de precio. 
  2. Adicionalmente, usa el método `.shift(1)` para traer el precio del día anterior a la fila actual y crea una columna `cambio_diario` que sea la resta del precio actual menos el precio anterior.

## Ejercicio 7: Manejo Avanzado de MultiIndex (`set_index`, `loc`) 🗂️
* **Contexto:** Tienes un reporte masivo de sensores IoT geolocalizados y quieres hacer consultas ultra rápidas por jerarquía geográfica.
* **Tarea:** Crea un DataFrame con `pais`, `ciudad`, `sensor_id`, y `temperatura`.
* **Requisito:** 1. Usa `set_index` para crear un MultiIndex jerárquico usando las columnas `['pais', 'ciudad']`. 
  2. Ordena el índice con `sort_index()` para optimizar búsquedas.
  3. Utilizando exclusivamente el localizador `.loc` (sin crear máscaras booleanas), extrae únicamente los registros de temperatura de una tupla específica, por ejemplo `('Perú', 'Lima')`.

In [67]:
N=10000

#Generamos las lista de paises de manera correcta
paises = rng.choice(["Perú","Chile","España","Brasil"],size=N)
# Lista para almacenar las ciudades
ciudades = []

for pais in paises:
    if pais == "Perú":
        ciudad = rng.choice(["Lima","Arequipa"])
    elif pais == "Chile":
        ciudad = rng.choice(["Santiago","Concepción"])
    elif pais == "España":
        ciudad = rng.choice(["Madrid","Valencia"])
    elif pais == "Brasil":
        ciudad = rng.choice(["Sao Paulo","Rio de Janeiro"])
    else:
        pd.na
    
    ciudades.append(ciudad)

# Crear el DataFrame
data = pd.DataFrame({
    "pais":paises,
    "ciudad":ciudades,
    "sensor_id":rng.choice(N,size=N,replace=False), #Nos aseguramos que los índices no se repiten
    "temperatura":rng.uniform(0,5000, size=N)
})

data.head()

,pais,ciudad,sensor_id,temperatura
0,Brasil,Rio de Janeiro,4533,"4,722.2905"
1,España,Valencia,9646,"1,595.5911"
2,Chile,Santiago,3297,"1,467.0837"
3,Perú,Lima,3811,"4,456.9490"
4,España,Madrid,1688,539.9592


In [68]:
reporte = (
    data
    .set_index(["pais","ciudad"])
    .sort_index()
)
reporte

sensor_id  temperatura
pais   ciudad                                
Brasil Rio de Janeiro       4533   4,722.2905
       Rio de Janeiro       3944   1,556.0950
       Rio de Janeiro       6578     288.2792
       Rio de Janeiro       5571     713.5070
       Rio de Janeiro       4489   3,680.1222
...                          ...          ...
Perú   Lima                 5933     967.0998
       Lima                 8807   1,763.4464
       Lima                 1507   2,117.1051
       Lima                 7919   3,550.6863
       Lima                 6986   4,287.8213

[10000 rows x 2 columns]

In [78]:
reporte.loc[("Perú","Lima")]

sensor_id  temperatura
pais ciudad                        
Perú Lima         3811   4,456.9490
     Lima         3986     910.6856
     Lima         8006   3,480.3578
     Lima         8348   2,501.1017
     Lima         3651   3,227.9106
...                ...          ...
     Lima         5933     967.0998
     Lima         8807   1,763.4464
     Lima         1507   2,117.1051
     Lima         7919   3,550.6863
     Lima         6986   4,287.8213

[1242 rows x 2 columns]

## Ejercicio 8: Discretización de Datos (`pd.cut` o `np.where`) 🏷️
* **Contexto:** RRHH quiere clasificar a los empleados en diferentes bandas salariales (categóricas) basándose en una columna numérica continua.
* **Tarea:** Crea un DataFrame con `empleado`, `departamento` y `salario`.
* **Requisito:** Crea una nueva columna `banda_salarial`. Utiliza la función `pd.cut()` de Pandas (o anidaciones de `np.where` si prefieres integrar NumPy) para dividir la columna salarios en 3 categorías discretas: "Junior" (< 2000), "Mid" (2000 a 5000), y "Senior" (> 5000).

In [98]:
#Numero de empleados
N=10000
df = pd.DataFrame({
    "empleado_id":rng.choice(N,size=N,replace=False),
    "departamento":rng.choice(["Lima","Callao","Trujillo","Arequipa","Cuzco"],size=N),
    "salario":rng.uniform(1000,10000,size=N)
})
df.head()

,empleado_id,departamento,salario
0,9852,Trujillo,"5,075.8258"
1,8273,Arequipa,"6,523.8496"
2,3494,Callao,"5,107.2526"
3,1682,Callao,"2,017.1287"
4,8091,Trujillo,"8,463.5641"


In [109]:
salarios = df["salario"].astype(float)
condiciones =[
            salarios < 2000,
            (2000 <= salarios) & (salarios <= 5000),
            salarios > 5000
             ]
categorías = ["Junior","Mid","Senior"]
df["banda_salarial"]= np.select(condiciones,categorías, default="Revisar")
df

,empleado_id,departamento,salario,banda_salarial
0,9852,Trujillo,"5,075.8258",Senior
1,8273,Arequipa,"6,523.8496",Senior
2,3494,Callao,"5,107.2526",Senior
3,1682,Callao,"2,017.1287",Mid
4,8091,Trujillo,"8,463.5641",Senior
...,...,...,...,...
9995,9086,Callao,"7,114.8825",Senior
9996,8009,Arequipa,"5,134.8142",Senior
9997,4080,Arequipa,"3,460.4886",Mid
9998,9964,Lima,"2,737.7644",Mid
